# VAE-GMM training on Colab: main model V1 (L = 64, β = 1e-4, critic on) on encounter 191, 150 epochs

Reproduction of Park et al., IEEE Access 14 (2026). The model code is the project's tested package (`whistle_code.zip`);
this notebook only sets up the session and runs it.

**Before the first session** (on the laptop, from the project root): `python colab/make_colab_bundle.py` writes
`colab/dist/whistle_code.zip`. Upload it to `whistle_vaegmm_E191/code/` in Google Drive, and the dataset to
`whistle_vaegmm_E191/data/preprocessed/`: `dataset_info.json` plus `encounter_191/images.npy` and `images.csv`
(the Drive folder is set in cell 1; cells 3 and 5 check these files).

**Every session:** run cells 1–6, then cell 8b (pre-flight check, ~2 min, trains nothing), then cell 9 (train).
**First session only:** cells 7–8 (checks + 3-min timing benchmark); cell 7 is also worth running on a GPU type it has not passed on.
After a disconnect, rerun cells 1–6, 8b and 9; training continues from the last finished epoch (checkpoints are on Drive).

Runtime: *Runtime → Change runtime type → A100 GPU*. Colab Pro has no background execution: keep this tab open.

In [ ]:
# ===== 1. Settings (the only cell to edit) =====
DRIVE_PROJECT   = "/content/drive/MyDrive/whistle_vaegmm_E191"   # the folder uploaded to Google Drive
RUN_ID          = "V1_L64_b1e-4_critic_E191"                # fallback run: "V1_L64_b1e-4_critic_E191_B896" with BATCH = 896
BATCH           = 5772     # 80 GB A100: 11 updates per epoch on encounter 191; "auto" = 5376 on an 80 GB A100, 2688 on a 40 GB A100, else 896 (paper); or an integer
MICRO_BATCH     = 2886     # images per pass (same maths as the whole batch). 2886 keeps every tensor under 2^31 numbers: the whole batch
                           # (0) gave wrong results on a G4 GPU on 2026-11-18. Out of GPU memory in cell 8b: 1443, then 962
TF32            = False    # True = faster tensor-core mode (small precision deviation, recorded in the run)
CUDNN_BENCHMARK = True     # cuDNN autotuning: same maths, faster kernels
CU_PER_HOUR     = 11.8     # approximate A100 compute units per hour (check your Colab plan page)

PAPER_BATCH, PAPER_LR_VAE, PAPER_LR_CRITIC = 896, 1e-3, 2e-4   # paper, App. A

In [ ]:
# ===== 2. GPU, RAM, Python; batch size and learning rates =====
import sys, subprocess, psutil, torch
assert sys.version_info >= (3, 11), f"needs Python >= 3.11, this runtime has {sys.version}"
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > A100 GPU"
props = torch.cuda.get_device_properties(0)
gpu_gib = props.total_memory / 2**30
print(f"GPU {props.name} {gpu_gib:.1f} GiB | RAM {psutil.virtual_memory().total / 2**30:.1f} GiB | "
      f"Python {sys.version.split()[0]} | torch {torch.__version__}")
if BATCH == "auto":
    BATCH_SIZE = 5376 if gpu_gib >= 79 else 2688 if gpu_gib >= 39 else PAPER_BATCH
else:
    BATCH_SIZE = int(BATCH)
LR_VAE = PAPER_LR_VAE * BATCH_SIZE / PAPER_BATCH          # linear scaling rule (no deviation at 896)
LR_CRITIC = PAPER_LR_CRITIC * BATCH_SIZE / PAPER_BATCH
print(f"batch {BATCH_SIZE} | lr VAE {LR_VAE:.6g} | lr critic {LR_CRITIC:.6g}")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
# ===== 3. Google Drive =====
import os
from google.colab import drive
drive.mount("/content/drive")
for rel in ("code/whistle_code.zip", "data/preprocessed/dataset_info.json"):
    assert os.path.exists(f"{DRIVE_PROJECT}/{rel}"), f"missing in Drive: {DRIVE_PROJECT}/{rel}"
print("Drive OK:", sorted(os.listdir(DRIVE_PROJECT)))

In [ ]:
# ===== 4. Code: unpack a fresh copy every session =====
import importlib.util, json, shutil, zipfile
CODE = "/content/whistle_code"
shutil.rmtree(CODE, ignore_errors=True)
with zipfile.ZipFile(f"{DRIVE_PROJECT}/code/whistle_code.zip") as z:
    z.extractall("/content")
if importlib.util.find_spec("soundfile") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "soundfile"], check=True)
if f"{CODE}/src" not in sys.path:
    sys.path.insert(0, f"{CODE}/src")
version = json.load(open(f"{CODE}/CODE_VERSION.json"))
print({k: v for k, v in version.items() if k != "files_sha256"})

In [ ]:
# ===== 5. Data: copy from Drive to the fast local disk (files already copied are skipped) =====
import glob, time
SRC, DST = f"{DRIVE_PROJECT}/data/preprocessed", "/content/preprocessed"
from whistlepipe.vaegmm.config import load_vaegmm_config
ENCOUNTERS = load_vaegmm_config(f"{CODE}/configs/vaegmm.toml")["deviations"]["encounters"]   # [deviations].encounters
needed = ["dataset_info.json"] + [f"encounter_{e}/{f}" for e in ENCOUNTERS for f in ("images.npy", "images.csv")]
t0 = time.time()
for rel in needed:
    s, d = f"{SRC}/{rel}", f"{DST}/{rel}"
    if not os.path.exists(s):
        raise FileNotFoundError(f"missing in Drive: data/preprocessed/{rel}")
    if os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s):
        continue
    os.makedirs(os.path.dirname(d), exist_ok=True)
    shutil.copyfile(s, d)
    print(f"copied {rel} ({os.path.getsize(d) / 1e9:.2f} GB)")
from whistlepipe.vaegmm.data import WhistleImages
images = WhistleImages(DST, ENCOUNTERS)        # checks every selected array against dataset_info.json
print(f"{len(images):,} images from {len(images.enc_dirs)} encounters ready ({time.time() - t0:.0f} s)")

In [ ]:
# ===== 6. Run configuration (paper values are locked inside configs/vaegmm.toml) =====
from whistlepipe.vaegmm.config import load_vaegmm_config
from whistlepipe.vaegmm.train import run_spec
steps_per_epoch = -(-len(images) // BATCH_SIZE)
overrides = [
    f'paths.dataset_dir="{DST}"',
    f'paths.runs_dir="{DRIVE_PROJECT}/runs"',
    f"deviations.batch_size={BATCH_SIZE}",
    f"deviations.lr_vae={LR_VAE!r}",
    f"deviations.lr_critic={LR_CRITIC!r}",
    'train.device="cuda"',
    "train.prefetch=2",
    f"train.log_every={max(1, steps_per_epoch // 4)}",
    f"train.micro_batch={MICRO_BATCH}",
    f"train.cudnn_benchmark={str(CUDNN_BENCHMARK).lower()}",
    f"train.tf32={str(TF32).lower()}",
]
cfg = load_vaegmm_config(f"{CODE}/configs/vaegmm.toml", overrides)
spec = run_spec(cfg, RUN_ID)                   # paper main model (L 64, beta 1e-4, critic); epochs from [deviations]: 150
print(json.dumps(spec, indent=1))
print("deviations:", cfg.to_dict()["deviations"], "| steps per epoch:", steps_per_epoch)

In [ ]:
# ===== 7. First session: data integrity (G1) and GPU numerics (G2), ~2 min =====
from whistlepipe.vaegmm.gates import gate_data, gate_numerics
g1, g2 = gate_data(cfg), gate_numerics(cfg)
assert g1["passed"] and g2["passed"], f"check failed, see {DRIVE_PROJECT}/runs/_gates"

In [ ]:
# ===== 8. First session: timing benchmark (~3 min, ~0.6 compute units; its model is discarded) =====
from whistlepipe.detector.ridgenet import set_deterministic
from whistlepipe.vaegmm.data import Prefetcher, epoch_plan
from whistlepipe.vaegmm.train import Trainer, apply_runtime_flags
set_deterministic()
apply_runtime_flags(cfg, spec)
tr = Trainer({**spec, "run_id": "benchmark"}, images, torch.device("cuda"), MICRO_BATCH)
vb, cb = epoch_plan(len(images), BATCH_SIZE, spec["n_critic"], spec["seed"], 1)
WARM, MEAS = 3, min(12, len(vb) - 3)          # one epoch may have fewer than 15 batches
pf = Prefetcher(images, vb[:WARM + MEAS], cb[:WARM + MEAS], depth=2)
g = tr.epoch_generator(1)
torch.cuda.reset_peak_memory_stats()
try:
    for i, (cbs, xb) in enumerate(pf):
        if i == WARM:
            torch.cuda.synchronize()
            t0, w0 = time.perf_counter(), pf.wait_s
        tr.step(cbs, xb, g)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
except torch.cuda.OutOfMemoryError:
    raise RuntimeError(f"out of GPU memory at batch {BATCH_SIZE}: set MICRO_BATCH = {BATCH_SIZE // 2} in cell 1, rerun cells 1, 2, 6 and 8")
finally:
    pf.close()
s_it = elapsed / MEAS
hours = s_it * steps_per_epoch * spec["epochs"] / 3600
peak = torch.cuda.max_memory_allocated() / 2**30
print(f"{s_it:.2f} s/iteration | {s_it * steps_per_epoch / 60:.1f} min/epoch | {hours:.1f} h for {spec['epochs']} epochs "
      f"| ~{hours * CU_PER_HOUR:.0f} compute units")
print(f"peak GPU memory {peak:.1f} of {gpu_gib:.1f} GiB | data-loader wait {100 * (pf.wait_s - w0) / elapsed:.1f} % of the time")
if peak > 0.93 * gpu_gib:
    print("WARNING: close to the memory limit; consider MICRO_BATCH = BATCH_SIZE // 2")
del tr
torch.cuda.empty_cache()

In [ ]:
# ===== 8b. Every session, right before cell 9: pre-flight check (~2 min; measures one step, trains nothing) =====
# Computes the first step of the next epoch from last.pt with the optimizer steps switched off: once with MICRO_BATCH (what
# cell 9 will use) and once in chunks of 192 images (the laptop setting that matched the A100 on 2026-11-18). No layer uses
# batch statistics, so the chunk size cannot change the maths: both must give the same rec and gradients. On 2026-11-18 a G4
# GPU with the whole batch of 5772 in one pass gave rec 212 instead of 172 on its first step and blew up three epochs later.
import datetime, hashlib
import numpy as np
from pathlib import Path
from whistlepipe.detector.ridgenet import set_deterministic
from whistlepipe.vaegmm.data import Prefetcher, epoch_plan
from whistlepipe.vaegmm.train import Trainer, apply_runtime_flags

LAST = Path(f"{DRIVE_PROJECT}/runs/{RUN_ID}/checkpoints/last.pt")
ck_epoch = int(torch.load(LAST, map_location="cpu", weights_only=False)["epoch"])
ck_sha = hashlib.sha256(LAST.read_bytes()).hexdigest()
print(f"last.pt: epoch {ck_epoch}, SHA-256 {ck_sha[:16]}")
assert ck_epoch != 101 or ck_sha.startswith("fcb1a2e6"), "last.pt says epoch 101 but is not the good epoch-101 checkpoint (fcb1a2e6...)"
set_deterministic()
apply_runtime_flags(cfg, spec)
DEV = torch.device("cuda")
VB, CB = epoch_plan(len(images), BATCH_SIZE, spec["n_critic"], spec["seed"], ck_epoch + 1)


def measure_first_step(micro):
    """First training step of the next epoch from last.pt (same batches and random draws as cell 9), nothing updated."""
    tr = Trainer(spec, images, DEV, micro)
    tr.load(LAST)
    tr.opt_v.step = tr.opt_c.step = lambda *a, **k: None          # optimizer steps off: weights and Adam state stay as loaded
    pf = Prefetcher(images, VB[:1], CB[:1], depth=1)
    try:
        cbs, xb = next(iter(pf))
        r = tr.step(cbs, xb, tr.epoch_generator(ck_epoch + 1), measure=True)
    finally:
        pf.close()
    vals = {k: float(v) for k, v in r.items()}
    grads = {"vae": torch.cat([p.grad.flatten() for p in tr.vae.parameters()]).double().cpu(),
             "critic": torch.cat([p.grad.flatten() for p in tr.critic.parameters()]).double().cpu()}
    del tr, r, cbs, xb
    torch.cuda.empty_cache()
    return vals, grads


def difference(a, b):
    return {"rec_diff": abs(a[0]["rec"] - b[0]["rec"]),
            **{f"grad_{k}_rel_diff": float((a[1][k] - b[1][k]).norm() / b[1][k].norm()) for k in a[1]}}


def agree(d):
    return d["rec_diff"] <= 0.05 and d["grad_vae_rel_diff"] <= 1e-3 and d["grad_critic_rel_diff"] <= 1e-3


t0 = time.time()
ref = measure_first_step(192)
use = measure_first_step(MICRO_BATCH)
d_use = difference(use, ref)
report = {"utc": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"), "gpu": props.name,
          "torch": torch.__version__, "micro_batch": MICRO_BATCH, "checkpoint_epoch": ck_epoch, "checkpoint_sha256": ck_sha,
          "rec_ref_192": ref[0]["rec"], "rec_micro_batch": use[0]["rec"], "gradnorm_vae_ref_192": ref[0]["gradnorm_vae"],
          "gradnorm_vae_micro_batch": use[0]["gradnorm_vae"], "diff_micro_batch_vs_192": d_use}
if MICRO_BATCH and gpu_gib >= 79:                    # diagnosis only: does the whole batch in one pass go wrong on this GPU?
    try:
        whole = measure_first_step(0)
        report.update({"rec_whole_batch": whole[0]["rec"], "diff_whole_batch_vs_192": difference(whole, ref)})
        del whole
    except torch.cuda.OutOfMemoryError:
        report["rec_whole_batch"] = "out of memory"
    torch.cuda.empty_cache()
PREFLIGHT_OK = bool(agree(d_use) and 150.0 < ref[0]["rec"] < 200.0)
PREFLIGHT_MICRO = MICRO_BATCH
report["passed"] = PREFLIGHT_OK
out = Path(f"{DRIVE_PROJECT}/runs/{RUN_ID}/preflight_{report['utc'].replace(':', '').replace('-', '')}.json")
out.write_text(json.dumps(report, indent=1), encoding="utf-8")
print(json.dumps(report, indent=1))
print(f"({time.time() - t0:.0f} s, saved to {out.name})")
if "diff_whole_batch_vs_192" in report:
    print("whole batch on this GPU:", "AGREES" if agree(report["diff_whole_batch_vs_192"]) else "WRONG (the 2026-11-18 cause)")
assert 150.0 < ref[0]["rec"] < 200.0, "rec of last.pt is far from the healthy ~172: wrong or damaged checkpoint, do not train"
assert PREFLIGHT_OK, f"MICRO_BATCH = {MICRO_BATCH} does not give the same step as chunks of 192 on this GPU: do not train with it"
print("PRE-FLIGHT PASSED: run cell 9")

### Cell 9: training

* Trains one epoch at a time. After each epoch its summary is compared with the median of the 10 epochs before it (the
  `check_final_epochs` rule: rec at most 5 % higher, KL at most 2x). A broken epoch is rolled back (`last.pt` goes back to the
  epoch before, kept as `last_good.pt`) and the run stops with an error, so a blow-up can no longer overwrite the last good
  checkpoint (it did on 2026-11-18). A non-finite value inside an epoch stops the run before anything is saved.
* Prints a few log lines per epoch and one epoch summary.
* Checkpoints: `runs/<RUN_ID>/checkpoints/last.pt` (every epoch), `epoch_025.pt` … (every 25), `final.pt` (end).
* **Resume:** rerun cells 1–6, then 8b and 9. Keep `BATCH = 5772` (the run refuses to continue with a different configuration).
  `MICRO_BATCH` only changes memory use and may differ between sessions; cell 8b checks it on the GPU you got.
* **Healthy start after epoch 101:** first log line at step 1113 with rec about 172 and gradnorm_vae about 220–240; the
  epoch summaries then fall slowly (171.2 at epoch 101, about 0.3 per epoch).
* **To see the curves:** stop cell 9, run cell 10, then run 8b and 9 again. It continues from the last finished epoch.
* **After an error** (out of memory, non-finite loss, rolled-back epoch): use *Runtime → Restart session*. Do not simply rerun:
  check the log first. Python keeps the failed step's data in GPU memory until the restart.

In [ ]:
# ===== 9. Train (resumable): one epoch at a time; a broken epoch is rolled back and the run stops =====
import gc
from whistlepipe.vaegmm.train import train_vae
assert globals().get("PREFLIGHT_OK") and PREFLIGHT_MICRO == MICRO_BATCH == int(cfg["train"]["micro_batch"]), \
    "run cells 1-6 and the pre-flight cell 8b first (it must pass, with the same MICRO_BATCH)"
RUN = Path(f"{DRIVE_PROJECT}/runs/{RUN_ID}")
CK = RUN / "checkpoints"


def epoch_summaries():
    ep = {}
    for line in open(RUN / "train_log.jsonl", encoding="utf-8"):
        r = json.loads(line)
        if r.get("epoch_summary"):
            ep[r["epoch"]] = r                       # an epoch logged twice: the latest line counts
    return ep


def epoch_ok(ep, e, window=10, rec_ratio=1.05, kl_ratio=2.0):
    """The check_final_epochs rule for epoch e: rec and KL against the median of the `window` epochs before it."""
    before = [ep[i] for i in range(e - window, e) if i in ep]
    rec0, kl0 = float(np.median([r["rec"] for r in before])), float(np.median([r["kl"] for r in before]))
    r = ep[e]
    return bool(np.isfinite(r["rec"]) and np.isfinite(r["kl"]) and r["rec"] <= rec_ratio * rec0 and r["kl"] <= kl_ratio * kl0), rec0, kl0


try:
    while True:
        done = int(torch.load(CK / "last.pt", map_location="cpu", weights_only=False)["epoch"])
        if done >= spec["epochs"]:
            break
        shutil.copyfile(CK / "last.pt", CK / "last_good.pt")          # the checkpoint before this epoch
        train_vae(cfg, spec, resume=True, images=images, stop_after_epoch=done + 1)
        ok, rec0, kl0 = epoch_ok(epoch_summaries(), done + 1)
        if not ok:
            shutil.copyfile(CK / "last_good.pt", CK / "last.pt")
            raise RuntimeError(f"epoch {done + 1} looks broken (median of the 10 epochs before: rec {rec0:.1f}, KL {kl0:.1f}); "
                               f"last.pt is back at epoch {done}. Check the log before training again.")
    print("finished:", json.load(open(RUN / "train_info.json"))["status"])
except KeyboardInterrupt:                      # handled here so that no stored traceback keeps the batches in GPU memory
    gc.collect()
    torch.cuda.empty_cache()
    print("stopped: rerun cells 8b and 9 to continue from the last finished epoch")

In [ ]:
# ===== 10. Progress: loss curves, time left, latest sample grid (run when training is stopped or finished) =====
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
run = f"{DRIVE_PROJECT}/runs/{RUN_ID}"
recs = [json.loads(l) for l in open(f"{run}/train_log.jsonl")]
ep = pd.DataFrame([r for r in recs if r.get("epoch_summary")]).drop_duplicates("epoch", keep="last")
cols = [c for c in ("rec", "kl", "adv", "loss_d", "d_real", "d_fake", "gp") if c in ep]
ep.plot(x="epoch", y=cols, subplots=True, layout=(-1, 2), figsize=(12, 2 * ((len(cols) + 1) // 2)))
plt.tight_layout()
plt.show()
done, per = int(ep.epoch.max()), float(ep.epoch_s.tail(5).mean())
left_h = (spec["epochs"] - done) * per / 3600
print(f"epoch {done}/{spec['epochs']} | {per / 60:.1f} min/epoch | ~{left_h:.1f} h left (~{left_h * CU_PER_HOUR:.0f} compute units)")
pngs = sorted(glob.glob(f"{run}/samples_epoch_*.png"))
if pngs:
    display(Image(pngs[-1]))

In [ ]:
# ===== 11. After training: check the run, encode the latents (~1 min) =====
from whistlepipe.vaegmm.evaluate import encode
from whistlepipe.vaegmm.train import check_final_epochs
info = json.load(open(f"{DRIVE_PROJECT}/runs/{RUN_ID}/train_info.json"))
print(info["status"], "| final checkpoint SHA-256:", info.get("final_checkpoint_sha256"))
health = check_final_epochs(f"{DRIVE_PROJECT}/runs/{RUN_ID}")
print(health)
if health["healthy"]:
    print(encode(cfg, RUN_ID))                 # latents_mu.npy (63,490 x 64): the GMM input
else:
    print("NOT ENCODED: the last epoch looks like a loss spike. Compare it with the earlier epochs in cell 10 before fitting the GMM.")

In [ ]:
# ===== 12. OPTIONAL: GMM fit here (~15 min, ~3 compute units). Default plan: fit it on the laptop (0 units). =====
RUN_GMM_HERE = False
if RUN_GMM_HERE:
    from whistlepipe.vaegmm.config import main_k
    from whistlepipe.vaegmm.evaluate import fit_gmm
    print(fit_gmm(cfg, RUN_ID, main_k(cfg)))

### After training

Download `whistle_vaegmm_E191/runs/V1_L64_b1e-4_critic_E191/` from Drive into the project's `models/vaegmm/` folder on the laptop,
then run the GMM fit and evaluation there (0 compute units):

```
python whistle.py fit-gmm --experiment V1 --K 3000
python whistle.py vae-eval recon --experiment V1
```